# 모델 배포(Model Deployment Overview)


## 배포 파이프라인 개요

```
①  최종 모델 학습 (K-Fold 앙상블)
        │ joblib.dump()
②  모델 파일 저장 (final_models/*.pkl)
        │
③  Streamlit 앱 (app.py)
        ├─ 사용자: CSV 업로드
        ├─ 모델: K개 예측 평균 → 최종 클래스
        └─ 결과: 성능 리포트 + 시각화 출력
```

---
## 0. 환경 설정

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
import os
import time

from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import (
    f1_score, roc_auc_score,
    classification_report, confusion_matrix
)
from sklearn.preprocessing import label_binarize
import warnings

warnings.filterwarnings('ignore')
plt.rcParams['axes.unicode_minus'] = False
%matplotlib inline

print("라이브러리 로드 완료")

In [ ]:
train_df = pd.read_csv('../00_data/train_processed.csv')

TARGET_COL = 'Y_Class'
META_COLS  = ['PRODUCT_ID', 'LINE', 'PRODUCT_CODE', 'Y_Class', 'Y_Quality']
feat_cols  = [c for c in train_df.columns if c not in META_COLS]

X = train_df[feat_cols]
y = train_df[TARGET_COL]

print(f"학습 데이터: {X.shape[0]:,} 행 × {X.shape[1]:,} 피처")
print(f"   Y_Class 분포:")
print(y.value_counts().sort_index())

---
## 1. K-Fold 앙상블 원리

### 왜 K-Fold 앙상블을 사용하는가?

```
Fold 1 모델: 전체 데이터의 80%로 학습 → 예측 확률 p1
Fold 2 모델: 다른 80%로 학습           → 예측 확률 p2
       ...               ...
Fold K 모델: 또 다른 80%로 학습         → 예측 확률 pK
                                                ↓
                               최종 예측 = (p1 + p2 + ... + pK) / K
```

- **분산 감소**: 각 모델이 서로 다른 데이터로 학습 → 오류가 평균에서 상쇄됨
- **전체 데이터 활용**: 각 Fold에서 모든 데이터가 1번씩 검증에 사용됨
- **단일 모델보다 일반화 성능 우수**: Variance를 줄이고 안정성 향상

---
## 2. 최종 모델 학습 (5-Fold Random Forest)

06_Final_Experiments에서 선정된 최적 모델(Random Forest)을  
**전체 데이터 5-Fold**로 학습하고 저장합니다.

In [ ]:
# 모델 저장 디렉토리 생성
MODEL_DIR = 'final_models'
os.makedirs(MODEL_DIR, exist_ok=True)

# 하이퍼파라미터
RF_PARAMS = {
    'n_estimators': 200,
    'max_depth'   : 12,
    'n_jobs'      : -1,
    'random_state': 42,
}

N_FOLDS = 5
skf     = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=42)

fold_scores = []
saved_models = []

print(f"5-Fold 학습 시작 (Random Forest)")
print(f"파라미터: {RF_PARAMS}")
print()

for fold, (tr_idx, val_idx) in enumerate(skf.split(X, y), 1):
    t0 = time.time()
    X_tr, X_val = X.iloc[tr_idx], X.iloc[val_idx]
    y_tr, y_val = y.iloc[tr_idx], y.iloc[val_idx]

    model = RandomForestClassifier(**RF_PARAMS)
    model.fit(X_tr, y_tr)

    # 성능 평가
    y_pred = model.predict(X_val)
    y_prob = model.predict_proba(X_val)
    macro_f1  = f1_score(y_val, y_pred, average='macro')
    micro_auc = roc_auc_score(
        label_binarize(y_val, classes=sorted(y.unique())),
        y_prob, multi_class='ovr', average='macro'
    )
    elapsed = time.time() - t0

    fold_scores.append({'Fold': fold, 'Macro F1': macro_f1, 'Micro AUC': micro_auc})

    # 모델 저장
    model_path = os.path.join(MODEL_DIR, f'rf_fold_{fold}.pkl')
    joblib.dump(model, model_path)
    saved_models.append(model_path)

    print(f"   Fold {fold}: Macro F1={macro_f1:.4f}, Micro AUC={micro_auc:.4f}  ({elapsed:.0f}s) → 저장: {model_path}")

scores_df = pd.DataFrame(fold_scores)
print(f"\n   CV 평균 Macro F1 : {scores_df['Macro F1'].mean():.4f} ± {scores_df['Macro F1'].std():.4f}")
print(f"   CV 평균 Micro AUC: {scores_df['Micro AUC'].mean():.4f} ± {scores_df['Micro AUC'].std():.4f}")

In [ ]:
# Fold별 성능 시각화
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('5-Fold Cross-Validation: Final Model Performance', fontsize=14, fontweight='bold')

metrics = ['Macro F1', 'Micro AUC']
colors  = ['#4ECDC4', '#FF6B6B']

for i, (metric, color) in enumerate(zip(metrics, colors)):
    vals = scores_df[metric].values
    axes[i].bar(range(1, N_FOLDS + 1), vals, color=color, edgecolor='white', linewidth=1.5)
    axes[i].axhline(vals.mean(), color='navy', linestyle='--', linewidth=2,
                    label=f'Mean = {vals.mean():.4f}')
    axes[i].fill_between(range(0, N_FOLDS + 2),
                         vals.mean() - vals.std(), vals.mean() + vals.std(),
                         alpha=0.15, color='navy')
    for j, v in enumerate(vals):
        axes[i].text(j + 1, v + 0.001, f'{v:.4f}', ha='center', fontsize=10, fontweight='bold')
    axes[i].set_title(f'{metric} per Fold', fontsize=12)
    axes[i].set_xlabel('Fold')
    axes[i].set_ylabel(metric)
    axes[i].set_xticks(range(1, N_FOLDS + 1))
    axes[i].set_xlim(0.5, N_FOLDS + 0.5)
    axes[i].legend()

plt.tight_layout()
plt.show()

---
## 3. 메타데이터 저장 및 앙상블 검증

In [ ]:
# 메타데이터 저장 (app.py에서 사용)
metadata = {
    'model_type'      : 'RandomForest',
    'n_folds'         : N_FOLDS,
    'n_features'      : X.shape[1],
    'n_samples'       : X.shape[0],
    'feature_names'   : list(X.columns),
    'classes'         : sorted(y.unique().tolist()),
    'mean_cv_f1'      : float(scores_df['Macro F1'].mean()),
    'mean_cv_score'   : float(scores_df['Micro AUC'].mean()),
    'std_cv_score'    : float(scores_df['Micro AUC'].std()),
    'rf_params'       : RF_PARAMS,
}

metadata_path = os.path.join(MODEL_DIR, 'metadata.pkl')
joblib.dump(metadata, metadata_path)

print("[저장된 파일]")
for path in saved_models + [metadata_path]:
    size_kb = os.path.getsize(path) / 1024
    print(f"   {path}  ({size_kb:.1f} KB)")

In [ ]:
# 앙상블 예측 검증 (Fold 1 모델들 불러와 예측 평균)
print("앙상블 예측 검증:")
loaded_models = [joblib.load(p) for p in sorted(saved_models)]

# 소규모 검증 (임의 10% 샘플)
sample_idx   = np.random.choice(len(X), size=int(len(X) * 0.1), replace=False)
X_sample     = X.iloc[sample_idx]
y_sample     = y.iloc[sample_idx]

# 각 모델의 예측 확률 평균
prob_list   = [m.predict_proba(X_sample) for m in loaded_models]
ensemble_prob = np.mean(prob_list, axis=0)          # shape: (n_samples, n_classes)
ensemble_pred = np.argmax(ensemble_prob, axis=1)

# 성능
ens_f1  = f1_score(y_sample, ensemble_pred, average='macro')
ens_auc = roc_auc_score(
    label_binarize(y_sample, classes=sorted(y.unique())),
    ensemble_prob, multi_class='ovr', average='macro'
)

print(f"   샘플 수  : {len(y_sample)}")
print(f"   Macro F1 : {ens_f1:.4f}")
print(f"   Micro AUC: {ens_auc:.4f}")

In [ ]:
# 혼동 행렬 시각화
cm = confusion_matrix(y_sample, ensemble_pred)

fig, ax = plt.subplots(figsize=(7, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax,
            xticklabels=[f'Pred {c}' for c in sorted(y.unique())],
            yticklabels=[f'True {c}' for c in sorted(y.unique())])
ax.set_title('Ensemble Model: Confusion Matrix (10% Sample)', fontsize=13, fontweight='bold')
ax.set_xlabel('Predicted Label')
ax.set_ylabel('True Label')
plt.tight_layout()
plt.show()

print("\nClassification Report:")
print(classification_report(y_sample, ensemble_pred, target_names=[f'Class {c}' for c in sorted(y.unique())]))

---
## 4. Streamlit 앱 실행 방법

학습된 모델(`final_models/`)을 활용하는 예측 대시보드를 실행합니다:

```bash
# 07_Deployment 폴더에서 실행
cd 07_Deployment
streamlit run app.py
```

브라우저에서 `http://localhost:8501`에 접속하면 다음 기능을 사용할 수 있습니다:

| 탭 | 내용 |
|---|---|
| 🎯 메인 | 예측 결과 테이블 및 클래스 분포 |
| 📈 성능 리포트 | AUROC, F1, Confusion Matrix |
| 🔍 예측 분석 | 샘플별 예측 확률 상세 보기 |
| 💼 비즈니스 분석 | LINE/PRODUCT_CODE별 불량률 분석 |
| 🧠 xAI | 피처 중요도 및 예측 근거 |

In [ ]:
# app.py 실행 가능 여부 확인
required_files = [
    'app.py',
    *[f'final_models/rf_fold_{i}.pkl' for i in range(1, N_FOLDS + 1)],
    'final_models/metadata.pkl'
]
print("[배포 준비 체크리스트]")
for fpath in required_files:
    exists = os.path.exists(fpath)
    status = "✅" if exists else "❌"
    print(f"   {status} {fpath}")

all_ok = all(os.path.exists(f) for f in required_files)
if all_ok:
    print("\n모든 파일 준비 완료! 아래 명령어로 앱을 실행하세요:")
    print("   streamlit run app.py")
else:
    print("\n일부 파일이 누락되었습니다. 위 셀들을 순서대로 실행하세요.")

---
## 5. app.py 핵심 코드 구조 이해

아래 코드는 `app.py`의 핵심 로직입니다. 직접 실행하지 않고 **개념 이해용**으로 살펴봅니다.

In [ ]:
# ── app.py 핵심 구조 ──────────────────────
"""
# 1. 모델 로드
models = [joblib.load(f'final_models/rf_fold_{i}.pkl') for i in range(1, 6)]
metadata = joblib.load('final_models/metadata.pkl')

# 2. 사용자 CSV 업로드
uploaded_file = st.sidebar.file_uploader("CSV 파일을 업로드하세요")
test_df = pd.read_csv(uploaded_file)

# 3. 앙상블 예측 (핵심!)
prob_list = [model.predict_proba(X_test) for model in models]
final_prob = np.mean(prob_list, axis=0)           # 확률 평균
final_pred = np.argmax(final_prob, axis=1)         # argmax → 클래스

# 4. 결과 시각화 (Plotly 인터랙티브 차트)
fig = px.bar(pred_df, x='Y_Class', title='Predicted Class Distribution')
st.plotly_chart(fig)
"""